prompt : 
Génère-moi un dataset COMPLET en CSV de BMW Z3 1.9 d’occasion en France disponible sur les site de vente de voiture d'occasion français avec ces colonnes : titre,date_publication,annee,kilometrage_km,prix_eur,ville,type_vendeur ,url,date_csv et sans doublon

In [3]:
import pandas as pd
import re

# ── 1. Chargement des deux CSV sources ─────────────────────────────────────
df1 = pd.read_csv("data.csv", encoding="utf-8-sig")
df2 = pd.read_csv("20_02_2026.csv", encoding="utf-8")

# ── 2. Fusion et déduplication sur l'URL ───────────────────────────────────
df = pd.concat([df1, df2], ignore_index=True)
df["date_csv"] = pd.to_datetime(df["date_csv"], errors="coerce")
df = df.sort_values("date_csv", ascending=False)
df = df.drop_duplicates(subset=["url"], keep="first").reset_index(drop=True)

# ── 3. Normalisation de date_publication → format dd/mm/yyyy ───────────────
df["date_publication"] = pd.to_datetime(df["date_publication"], format="mixed", errors="coerce")
df["date_publication"] = df["date_publication"].dt.strftime("%d/%m/%Y")

# ── 4. Extraction de l'ID depuis l'URL ─────────────────────────────────────
df["id"] = df["url"].str.extract(r"/(\d+)$").astype("Int64")

# ── 5. Colonnes manquantes à ajouter (valeurs par défaut) ──────────────────
df["departement"]   = ""
df["code_postal"]   = ""
df["carburant"]     = "essence"
df["boite_vitesse"] = "manuelle"
df["portes"]        = 2

# ── 6. Harmonisation du type_vendeur (Professionnel → professionnel) ───────
df["type_vendeur"] = df["type_vendeur"].str.lower().str.strip()

# ── 7. Sélection et ordre des colonnes cibles ──────────────────────────────
colonnes_cibles = [
    "id", "titre", "date_publication", "annee", "kilometrage_km",
    "prix_eur", "ville", "departement", "code_postal",
    "type_vendeur", "carburant", "boite_vitesse", "portes", "url"
]
df = df[colonnes_cibles]

# ── 8. Nettoyage final ─────────────────────────────────────────────────────
df["annee"]          = pd.to_numeric(df["annee"], errors="coerce").astype("Int64")
df["kilometrage_km"] = pd.to_numeric(df["kilometrage_km"], errors="coerce").astype("Int64")
df["prix_eur"]       = pd.to_numeric(df["prix_eur"], errors="coerce").astype("Int64")

print(f"✅ {len(df)} annonces fusionnées")
print(df.dtypes)
df.head()

# ── 9. Export ──────────────────────────────────────────────────────────────
df.to_csv("bmw_z3_merged.csv", index=False, encoding="utf-8-sig")
print("✅ Fichier exporté : bmw_z3_merged.csv")


✅ 16 annonces fusionnées
id                   Int64
titre               object
date_publication    object
annee                Int64
kilometrage_km       Int64
prix_eur             Int64
ville               object
departement         object
code_postal         object
type_vendeur        object
carburant           object
boite_vitesse       object
portes               int64
url                 object
dtype: object
✅ Fichier exporté : bmw_z3_merged.csv
